# BellAI - Gestor de Entrenamiento en Google Colab
Menu interactivo con exactamente 5 modulos de BellAI, delegacion al orquestador central train.py y gestion de checkpoints en Google Drive.

In [ ]:
# 1. Montar Drive y Configurar Autenticacion de Kaggle
import os
import subprocess
from google.colab import drive

drive.mount('/content/drive')

# Reemplaza TU_KAGGLE_API_TOKEN_AQUI con tu token privado de Kaggle
os.environ['KAGGLE_API_TOKEN'] = 'TU_KAGGLE_API_TOKEN_AQUI'
!mkdir -p ~/.kaggle && echo $KAGGLE_API_TOKEN > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

# 2. Clonar/Actualizar el repositorio
if not os.path.exists('/content/modelIATrainer'):
    !git clone https://github.com/ldgd2/modelIATrainer.git
else:
    %cd /content/modelIATrainer
    !git pull

%cd /content/modelIATrainer
!pip install -q -r requirements.txt

print('\n' + '='*50)
print(' [*] BELLAI - GESTOR DE ENTRENAMIENTO MODULAR')
print('='*50)

# Los 5 modulos de BellAI
modulos = [
    ('armario_digital', 'Deteccion y Segmentacion de Prendas (DeepFashion2 -> wardrobe_v1.pt)'),
    ('diagnostico_morfologico', 'Diagnostico Facial y Corporal (FairFace, Keypoints, CelebAMask)'),
    ('motor_recomendacion', 'Motor de Recomendacion y Compatibilidad (Polyvore -> style_recommender_v1.pt)'),
    ('simulacion_virtual', 'Probador Virtual y Maquillaje (VITON-HD, MT-Dataset)'),
    ('asistente_conversacional', 'Asistente Conversacional LLM Local (Hugging Face -> bellai_assistant_8b.gguf)')
]

print('\nSelecciona el modulo a entrenar:')
for i, (nombre, desc) in enumerate(modulos, 1):
    print(f'  {i}. {nombre.ljust(26)} | {desc}')

idx = int(input('\nNumero de modulo (1-5): ')) - 1
modulo_sel = modulos[idx][0]

task_arg = ''
if modulo_sel == 'diagnostico_morfologico':
    print('\nSub-tarea para diagnostico_morfologico:')
    print('  1. colorimetria (FairFace -> colorimetry_v1.pt)')
    print('  2. morfologia   (COCO Keypoints -> morphology_v1.pt)')
    print('  3. capilar      (CelebAMask-HQ -> hair_analysis_v1.pt)')
    sub_idx = int(input('Selecciona sub-tarea (1-3): '))
    subtareas = ['colorimetria', 'morfologia', 'capilar']
    task_arg = f'--task {subtareas[sub_idx - 1]} --download_data'
elif modulo_sel == 'simulacion_virtual':
    print('\nSub-tarea para simulacion_virtual:')
    print('  1. probador   (VITON-HD -> virtual_tryon_v1.pt)')
    print('  2. maquillaje (MT-Dataset -> makeup_sim_v1.pt)')
    sub_idx = int(input('Selecciona sub-tarea (1-2): '))
    subtareas = ['probador', 'maquillaje']
    task_arg = f'--task {subtareas[sub_idx - 1]} --download_data'

resume_arg = ''
if modulo_sel != 'asistente_conversacional':
    print('\nModo de ejecucion:')
    print('  1. Iniciar nuevo entrenamiento (Desde cero)')
    print('  2. Continuar desde un checkpoint existente en Google Drive')
    modo = int(input('Selecciona una opcion (1-2): '))
    
    if modo == 2:
        ruta_checkpoints = f'/content/drive/MyDrive/Modelos_BellAI/{modulo_sel}/checkpoints'
        if os.path.exists(ruta_checkpoints):
            archivos_pt = [f for f in os.listdir(ruta_checkpoints) if f.endswith('.pt')]
            archivos_pt.sort()
            if len(archivos_pt) > 0:
                print('\nCheckpoints encontrados en Drive:')
                for i, ckpt in enumerate(archivos_pt, 1):
                    print(f'  {i}. {ckpt}')
                idx_ckpt = int(input('Numero de checkpoint a restaurar: ')) - 1
                ckpt_path = f'{ruta_checkpoints}/{archivos_pt[idx_ckpt]}'
                resume_arg = f'--resume "{ckpt_path}"'
                print(f'[*] Checkpoint seleccionado: {archivos_pt[idx_ckpt]}')
            else:
                print('[!] No se hallaron checkpoints .pt en esta carpeta. Se iniciara desde cero.')
        else:
            print('[!] Carpeta de checkpoints aun no existe. Se iniciara desde cero.')

# Construir comando final
extra_args = ''
if modulo_sel == 'armario_digital':
    extra_args = '--epochs 10 --save_every 2 --batch_size 64 --password "2019Deepfashion2**"'
elif modulo_sel == 'motor_recomendacion':
    extra_args = '--epochs 15 --save_every 2 --batch_size 32'

comando_final = f'python train.py --module {modulo_sel} {task_arg} {extra_args} {resume_arg}'.strip()

print('\n' + '='*50)
print('[*] Ejecutando comando:')
print(comando_final)
print('='*50 + '\n')

subprocess.run(comando_final, shell=True)
